In [4]:
# ##############################################################
# CELL — 4 plots only, LOADED FROM THE SAVED .npz:
# vertical chi, vertical delta chi (or orientation),
# horizontal chi, horizontal delta chi (or orientation)
#
# Updates in real time as you change ROI / tol / colormaps / orientation toggle.
# ##############################################################

%matplotlib widget

import os
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, FloatSlider, Dropdown, Checkbox, VBox, HBox, Output, interactive_output
from IPython.display import display, clear_output


# ============================================================
# Load the saved fibre maps  ->  batch_result
# ============================================================
NPZ_PATH = "/data/visitor/ihhg69/id13/20260409/PROCESSED_DATA/BLA/dq_modernpap/dq_modernpap_roi146587_164515/dq_modernpap_roi146587_164515_fiber_maps.npz"

print("Loading:", NPZ_PATH)
_d = np.load(NPZ_PATH, allow_pickle=True)

batch_result = {
    "chi1_maps":  _d["chi1_maps"],
    "dchi1_maps": _d["dchi1_maps"],
    "chi2_maps":  _d["chi2_maps"],
    "dchi2_maps": _d["dchi2_maps"],
    "npeaks_maps": _d["npeaks_maps"] if "npeaks_maps" in _d.files else None,
    "saved_rois": [tuple(r) for r in np.atleast_2d(_d["saved_rois"])],
    "n_rows": int(_d["n_rows"]),
    "n_cols": int(_d["n_cols"]),
}
print(f"Loaded {len(batch_result['saved_rois'])} ROI(s), "
      f"maps {batch_result['chi1_maps'].shape}")


# ============================================================
# Axial angle utilities
# ============================================================

def wrap_axial_chi(angle):
    """Axial wrapping: chi and chi + 180 deg are equivalent. -> [-90, 90)."""
    return (np.asarray(angle, dtype=float) + 90.0) % 180.0 - 90.0


def axial_diff(a, b):
    """Axial angular difference. -> [-90, 90)."""
    return (np.asarray(a, dtype=float) - np.asarray(b, dtype=float) + 90.0) % 180.0 - 90.0


def axial_abs_diff(a, b):
    """Absolute axial angular separation. -> [0, 90]."""
    return np.abs(axial_diff(a, b))


def robust_vlim(arr, low=2, high=98):
    """Robust colorbar limits using percentiles."""
    vals = arr[np.isfinite(arr)]

    if len(vals) == 0:
        return None, None

    vmin = np.nanpercentile(vals, low)
    vmax = np.nanpercentile(vals, high)

    if np.isclose(vmin, vmax):
        vmin = np.nanmin(vals)
        vmax = np.nanmax(vals)

    if np.isclose(vmin, vmax):
        vmin = vmin - 1
        vmax = vmax + 1

    return vmin, vmax


# ============================================================
# Sort fitted chi families into vertical / horizontal
# ============================================================

def sort_into_vertical_horizontal(
    chi1_map, dchi1_map, chi2_map, dchi2_map,
    vertical_ref=0.0, horizontal_ref=90.0, tol_deg=35.0
):
    """
    Sort fitted families into vertical / horizontal. The crossing region
    is included in both maps if both fitted orientations exist and each
    one belongs to a family.
    """

    vertical_ref = wrap_axial_chi(vertical_ref)
    horizontal_ref = wrap_axial_chi(horizontal_ref)

    nrows, ncols = chi1_map.shape

    chi_vertical = np.full((nrows, ncols), np.nan)
    dchi_vertical = np.full((nrows, ncols), np.nan)

    chi_horizontal = np.full((nrows, ncols), np.nan)
    dchi_horizontal = np.full((nrows, ncols), np.nan)

    for i in range(nrows):
        for j in range(ncols):

            candidates = []

            if np.isfinite(chi1_map[i, j]):
                candidates.append({"chi": chi1_map[i, j], "dchi": dchi1_map[i, j]})

            if np.isfinite(chi2_map[i, j]):
                candidates.append({"chi": chi2_map[i, j], "dchi": dchi2_map[i, j]})

            if len(candidates) == 0:
                continue

            best_v, best_v_dist = None, np.inf
            for cand in candidates:
                dist = axial_abs_diff(cand["chi"], vertical_ref)
                if dist < best_v_dist:
                    best_v_dist, best_v = dist, cand

            best_h, best_h_dist = None, np.inf
            for cand in candidates:
                dist = axial_abs_diff(cand["chi"], horizontal_ref)
                if dist < best_h_dist:
                    best_h_dist, best_h = dist, cand

            if best_v is not None and best_v_dist <= tol_deg:
                chi_vertical[i, j] = best_v["chi"]
                dchi_vertical[i, j] = best_v["dchi"]

            if best_h is not None and best_h_dist <= tol_deg:
                chi_horizontal[i, j] = best_h["chi"]
                dchi_horizontal[i, j] = best_h["dchi"]

    return chi_vertical, dchi_vertical, chi_horizontal, dchi_horizontal


def to_orientation(dchi_map):
    """Degree of orientation = (180 - FWHM) / 180, elementwise on a dchi map."""
    out = np.full_like(dchi_map, np.nan, dtype=float)
    mask = np.isfinite(dchi_map)
    out[mask] = (180.0 - dchi_map[mask]) / 180.0
    return out


# ============================================================
# Cache the sort step per (roi_id, tol_deg) to avoid recompute
# when only colormap/orientation toggles change
# ============================================================

_sort_cache = {}

def get_sorted_maps(roi_id, tol_deg):
    key = (roi_id, tol_deg)
    if key not in _sort_cache:
        iroi = roi_id - 1
        _sort_cache.clear()  # keep cache size at 1
        _sort_cache[key] = sort_into_vertical_horizontal(
            batch_result["chi1_maps"][iroi],
            batch_result["dchi1_maps"][iroi],
            batch_result["chi2_maps"][iroi],
            batch_result["dchi2_maps"][iroi],
            vertical_ref=0.0, horizontal_ref=90.0, tol_deg=tol_deg
        )
    return _sort_cache[key]


# ============================================================
# Plot 4 maps
# ============================================================

def plot_vertical_horizontal_maps(
    roi_id, tol_deg, chi_cmap, dchi_cmap, show_orientation,
    robust_low=2, robust_high=98
):
    roi_min, roi_max = batch_result["saved_rois"][roi_id - 1]

    chi_vertical_raw, dchi_vertical, chi_horizontal_raw, dchi_horizontal = get_sorted_maps(roi_id, tol_deg)

    # --- TRUE fitted-angle display (no reference subtraction) ---
    chi_vertical = chi_vertical_raw.copy()                          # [-90, 90)
    chi_horizontal = np.where(np.isfinite(chi_horizontal_raw),
                              chi_horizontal_raw % 180.0, np.nan)    # [0, 180)

    # --- delta chi panels, optionally converted to degree of orientation ---
    if show_orientation:
        panel_v = to_orientation(dchi_vertical)
        panel_h = to_orientation(dchi_horizontal)
        dchi_label = "orientation (0-1)"
        dchi_title_v, dchi_title_h = "Vertical: orientation", "Horizontal: orientation"
        vmin_dchi_v, vmax_dchi_v = 0.0, 1.0
        vmin_dchi_h, vmax_dchi_h = 0.0, 1.0
    else:
        panel_v = dchi_vertical
        panel_h = dchi_horizontal
        dchi_label = "deg"
        dchi_title_v, dchi_title_h = "Vertical: Δχ", "Horizontal: Δχ"
        vmin_dchi_v, vmax_dchi_v = robust_vlim(panel_v, low=robust_low, high=robust_high)
        vmin_dchi_h, vmax_dchi_h = robust_vlim(panel_h, low=robust_low, high=robust_high)

    vmin_chi_v, vmax_chi_v = robust_vlim(chi_vertical, low=robust_low, high=robust_high)
    vmin_chi_h, vmax_chi_h = robust_vlim(chi_horizontal, low=robust_low, high=robust_high)

    fig, axes = plt.subplots(2, 2, figsize=(9, 9))
    axes = axes.ravel()

    im0 = axes[0].imshow(chi_vertical, origin="upper", aspect="equal",
                         vmin=vmin_chi_v, vmax=vmax_chi_v, cmap=chi_cmap)
    axes[0].set_title("Vertical: χ")
    plt.colorbar(im0, ax=axes[0], label="deg")

    im1 = axes[1].imshow(panel_v, origin="upper", aspect="equal",
                         vmin=vmin_dchi_v, vmax=vmax_dchi_v, cmap=dchi_cmap)
    axes[1].set_title(dchi_title_v)
    plt.colorbar(im1, ax=axes[1], label=dchi_label)

    im2 = axes[2].imshow(chi_horizontal, origin="upper", aspect="equal",
                         vmin=vmin_chi_h, vmax=vmax_chi_h, cmap=chi_cmap)
    axes[2].set_title("Horizontal: χ")
    plt.colorbar(im2, ax=axes[2], label="deg")

    im3 = axes[3].imshow(panel_h, origin="upper", aspect="equal",
                         vmin=vmin_dchi_h, vmax=vmax_dchi_h, cmap=dchi_cmap)
    axes[3].set_title(dchi_title_h)
    plt.colorbar(im3, ax=axes[3], label=dchi_label)

    for ax in axes:
        ax.set_xticks([]); ax.set_yticks([])

    fig.suptitle(f"ROI {roi_id}: 2θ = {roi_min:.4f}–{roi_max:.4f}  (tol={tol_deg}°)", fontsize=13)

    fig.tight_layout()
    plt.show()


# ============================================================
# Interactive controls — live update, no button needed
# ============================================================

roi_slider = IntSlider(value=1, min=1, max=len(batch_result["saved_rois"]), step=1,
                        description="ROI", continuous_update=False)

tol_slider = FloatSlider(value=35.0, min=5.0, max=60.0, step=2.5,
                          description="tol deg", continuous_update=False)

CMAP_OPTIONS = ["twilight_shifted", "hsv", "viridis", "plasma", "coolwarm",
                "magma", "cividis", "turbo", "jet"]

chi_cmap_dd = Dropdown(options=CMAP_OPTIONS, value="twilight_shifted", description="χ cmap")
dchi_cmap_dd = Dropdown(options=CMAP_OPTIONS, value="viridis", description="Δχ cmap")

orientation_cb = Checkbox(value=False, description="Show as orientation (180-Δχ)/180")

out = Output()

def _update(roi_id, tol_deg, chi_cmap, dchi_cmap, show_orientation):
    with out:
        clear_output(wait=True)
        plt.close("all")
        plot_vertical_horizontal_maps(
            roi_id=roi_id, tol_deg=tol_deg,
            chi_cmap=chi_cmap, dchi_cmap=dchi_cmap,
            show_orientation=show_orientation
        )

linked = interactive_output(
    _update,
    {
        "roi_id": roi_slider,
        "tol_deg": tol_slider,
        "chi_cmap": chi_cmap_dd,
        "dchi_cmap": dchi_cmap_dd,
        "show_orientation": orientation_cb,
    }
)

display(VBox([
    HBox([roi_slider, tol_slider]),
    HBox([chi_cmap_dd, dchi_cmap_dd, orientation_cb]),
    out
]))

Loading: /data/visitor/ihhg69/id13/20260409/PROCESSED_DATA/BLA/dq_modernpap/dq_modernpap_roi146587_164515/dq_modernpap_roi146587_164515_fiber_maps.npz
Loaded 1 ROI(s), maps (1, 320, 160)
